Etapa 3 – Análisis exploratorio de datos (EDA)

In [ ]:
import re
import pandas as pd
import requests
from io import StringIO
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

BASE SINIESTROS

In [ ]:
def extract_drive_id(url_or_id: str) -> str:
    """
    Acepta un file_id directo o una URL de Drive y devuelve el file_id.
    Soporta:
      - https://drive.google.com/file/d/<ID>/view?usp=sharing
      - https://drive.google.com/open?id=<ID>
      - <ID> directo
    """
    s = url_or_id.strip()

    # Si parece un ID ya (sin http ni /)
    if "http" not in s and "/" not in s and len(s) > 20:
        return s

    # /file/d/<ID>/...
    m = re.search(r"/file/d/([a-zA-Z0-9_-]+)", s)
    if m:
        return m.group(1)

    # open?id=<ID>
    m = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", s)
    if m:
        return m.group(1)

    raise ValueError("No pude extraer el file_id. Pegá un link de archivo de Drive válido o el ID directamente.")

def read_gdrive_csv(url_or_id: str, sep_guess=","):
    file_id = extract_drive_id(url_or_id)
    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"

    resp = requests.get(download_url)
    resp.raise_for_status()
    head = resp.text[:600].lower()

    # Si Drive devolvió HTML (confirmación/permisos), avisamos
    if "<html" in head and "drive" in head:
        raise RuntimeError(
            "Google Drive devolvió HTML (posible verificación o permisos). "
            "Asegurate de que el archivo esté compartido con 'Cualquiera con el enlace (lector)'."
        )

    # Intento con autodetección y fallback de separador
    try:
        return pd.read_csv(StringIO(resp.text), sep=None, engine="python", encoding="utf-8")
    except Exception:
        try:
            return pd.read_csv(StringIO(resp.text), sep=sep_guess, encoding="utf-8")
        except Exception:
            return pd.read_csv(StringIO(resp.text), sep=";", encoding="latin-1")

# 👉 Pegá acá la URL del archivo (NO la de la carpeta)
url_del_archivo = "https://drive.google.com/file/d/1Ds9zXog-lPKlxFIopsBlWhLC45MUVX7o/view?usp=sharing"

Base_Siniestros = read_gdrive_csv(url_del_archivo)
print(Base_Siniestros.head(), Base_Siniestros.shape)

      siniestro     fec_ocu    poliza  item  endoso  tipo_stro  estado  \
0  60072907-2-1  2012-05-14  4-283519     1       1          0       0   
1  60080866-2-2  2012-07-01  4-340741     1       1          0       0   
2  60087362-1-2  2012-08-06  4-400748     1       1          0       0   
3  60087362-3-1  2012-08-06  4-400748     1       1          0       0   
4  60087636-2-2  2012-08-07  4-364604     1       2          0       0   

      pag_CASCO  pag_RC  pag_RCL  ...  inc_RT  inc_RP  inc_DT  inc_DP  inc_IT  \
0    129.712669     0.0        0  ...       0       0       0       0       0   
1   1445.193675     0.0        0  ...       0       0       0       0       0   
2  17751.260385     0.0        0  ...       0       0       0       0       0   
3    437.937477     0.0        0  ...       0       0       0       0       0   
4    122.577137     0.0        0  ...       0       0       0       0       0   

   inc_IP  inc_CR  inc_Otros     Inc_Pesos  Cobertura  
0       0   

In [ ]:
# Setup mínimo

pd.set_option("display.max_columns", 200) # Permite ver hasta 200 columnas cuando imprimís un DataFrame
pd.set_option("display.float_format", lambda x: f"{x:,.4f}") # Formatea números para que se vean prolijos



In [ ]:
# Tamaño + vista rápida

print("Shape (filas, columnas):", Base_Siniestros.shape)
display(Base_Siniestros.head(3))
display(Base_Siniestros.sample(min(3, len(Base_Siniestros)), random_state=42))


Shape (filas, columnas): (311175, 45)


,siniestro,fec_ocu,poliza,item,endoso,tipo_stro,estado,pag_CASCO,pag_RC,pag_RCL,pag_RCD,pag_RT,pag_RP,pag_DT,pag_DP,pag_IT,pag_IP,pag_CR,pag_Otros,rva_CASCO,rva_RC,rva_RCL,rva_RCD,rva_RT,rva_RP,rva_DT,rva_DP,rva_IT,rva_IP,rva_CR,rva_Otros,inc_CASCO,inc_RC,inc_RCL,inc_RCD,inc_RT,inc_RP,inc_DT,inc_DP,inc_IT,inc_IP,inc_CR,inc_Otros,Inc_Pesos,Cobertura
0,60072907-2-1,2012-05-14,4-283519,1,1,0,0,129.7127,0.0000,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,129.7127,0.0000,0,0,0,0,0,0,0,0,0,0,129.7127,Casco
1,60080866-2-2,2012-07-01,4-340741,1,1,0,0,"1,445.1937",0.0000,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,"1,445.1937",0.0000,0,0,0,0,0,0,0,0,0,0,"1,445.1937",Casco
2,60087362-1-2,2012-08-06,4-400748,1,1,0,0,"17,751.2604",0.0000,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,"17,751.2604",0.0000,0,0,0,0,0,0,0,0,0,0,"17,751.2604",Casco


,siniestro,fec_ocu,poliza,item,endoso,tipo_stro,estado,pag_CASCO,pag_RC,pag_RCL,pag_RCD,pag_RT,pag_RP,pag_DT,pag_DP,pag_IT,pag_IP,pag_CR,pag_Otros,rva_CASCO,rva_RC,rva_RCL,rva_RCD,rva_RT,rva_RP,rva_DT,rva_DP,rva_IT,rva_IP,rva_CR,rva_Otros,inc_CASCO,inc_RC,inc_RCL,inc_RCD,inc_RT,inc_RP,inc_DT,inc_DP,inc_IT,inc_IP,inc_CR,inc_Otros,Inc_Pesos,Cobertura
216768,60338298-1-1,2017-03-11,4-1337465,1,1,0,0,449.5239,0.0000,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,449.5239,0.0000,0,0,0,0,0,0,0,0,0,0,449.5239,Casco
62113,60115397-1-1,2013-02-06,4-496406,1,2,0,0,575.5471,0.0000,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,575.5471,0.0000,0,0,0,0,0,0,0,0,0,0,575.5471,Casco
134209,60223109-1-1,2015-03-19,4-799299,183,1,0,0,"2,854.6750",0.0000,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,"2,854.6750",0.0000,0,0,0,0,0,0,0,0,0,0,"2,854.6750",Casco


In [8]:
dtypes = Base_Siniestros.dtypes.value_counts()
print(dtypes)

display(Base_Siniestros.dtypes.sort_values())


int64      34
float64     7
object      4
dtype: int64


rva_RCD        int64
rva_RCL        int64
rva_RT         int64
rva_RP         int64
rva_DT         int64
rva_DP         int64
rva_IT         int64
rva_IP         int64
rva_CR         int64
rva_Otros      int64
inc_RCL        int64
inc_RCD        int64
inc_RT         int64
inc_RP         int64
inc_DT         int64
inc_DP         int64
inc_IT         int64
pag_Otros      int64
inc_IP         int64
pag_IP         int64
pag_IT         int64
inc_Otros      int64
pag_DP         int64
pag_DT         int64
pag_RP         int64
pag_RT         int64
pag_RCD        int64
pag_RCL        int64
pag_CR         int64
inc_CR         int64
item           int64
endoso         int64
estado         int64
tipo_stro      int64
rva_CASCO    float64
pag_RC       float64
inc_CASCO    float64
pag_CASCO    float64
Inc_Pesos    float64
inc_RC       float64
rva_RC       float64
siniestro     object
fec_ocu       object
poliza        object
Cobertura     object
dtype: object

In [20]:
missing_pct = (
    Base_Siniestros.isna()
    .mean()
    .sort_values(ascending=False) * 100
).round(2)

# IF para detectar si hay columnas con missing
if (missing_pct > 0).any():
    display(missing_pct.to_frame("missing_pct"))
else:
    print("No hay valores faltantes")

No hay valores faltantes


In [ ]:
print("Duplicados de fila completos:", Base_Siniestros.duplicated().sum())

Duplicados de fila completos: 0


In [ ]:
import pandas as pd

# 1) Agregación por Cobertura
tabla = (
    Base_Siniestros
        .groupby("Cobertura", dropna=False)
        .agg(
            total_Inc_Pesos=("Inc_Pesos", "sum"),
            cantidad_casos=("Inc_Pesos", "size")  # cuenta filas (casos)
        )
        .reset_index()
)

# 2) Totales generales
total_monto = tabla["total_Inc_Pesos"].sum()
total_casos = tabla["cantidad_casos"].sum()

# 3) Porcentajes
tabla["pct_Inc_Pesos"] = (tabla["total_Inc_Pesos"] / total_monto * 100).round(2)
tabla["pct_casos"] = (tabla["cantidad_casos"] / total_casos * 100).round(2)

# 4) Fila TOTAL al final
fila_total = pd.DataFrame({
    "Cobertura": ["TOTAL"],
    "total_Inc_Pesos": [total_monto],
    "cantidad_casos": [total_casos],
    "pct_Inc_Pesos": [100.00],
    "pct_casos": [100.00]
})

# 5) Ordenar por monto (y dejar TOTAL abajo)
tabla_sin_total = tabla.sort_values("total_Inc_Pesos", ascending=False)
tabla_final = pd.concat([tabla_sin_total, fila_total], ignore_index=True)




display(tabla_final)

,Cobertura,total_Inc_Pesos,cantidad_casos,pct_Inc_Pesos,pct_casos
0,RC,"9,074,254,920.3381",132401,72.2700,42.5500
1,Casco,"3,481,729,042.5141",178004,27.7300,57.2000
2,S/C,0.0000,770,0.0000,0.2500
3,TOTAL,"12,555,983,962.8522",311175,100.0000,100.0000


In [ ]:
Base_Siniestros["poliza_iteml"] = (
    Base_Siniestros["poliza"].astype(str) +
    Base_Siniestros["item"].astype(str)
)


Base_Siniestros["poliza_iteml"].value_counts()[Base_Siniestros["poliza_iteml"].value_counts() > 1]

4-16358252901    31
4-16358254161    30
4-15590471       17
4-15465312850    15
4-10811533264    14
                 ..
4-1009721         2
4-4606861         2
4-161072916       2
4-21001514        2
4-1082855         2
Name: poliza_iteml, Length: 55734, dtype: int64